Cloud-only data dependency: this notebook expects access to the PROFILE_DATA clinical-note parquets, and either Azure AD credentials (`dfci_gpt`) or Application Default Credentials with Vertex AI permission (`vertex_ai`), depending on `PROVIDER` below.

# AR-low / AR-indifferent (ARLPC) Pipeline Runner

Labels each cohort patient as ever AR-low / AR-indifferent, in two tiers, independent of
NE status and disease state:
- **strict**: AR-low tissue IHC (negative or any reduced wording), AR-specific statements
  (AR-negative/null/low/indifferent, double-negative/DNPC), somatic genomic AR loss;
- **broad**: strict plus androgen-independent / hormone-refractory wording.

AR-target IHC loss (PSA/PSAP/NKX3.1/PSMA/HOXB13) is corroborating only; AR-positive IHC is
audit only. Each tier carries its earliest date (stated in the text, else the note date),
plus a separate earliest AR-low tissue date. A deterministic gate (`tasks/arlpc/ar_gate.py`)
decides every finding.

Pipeline steps:
1. Scan Pathology (whole reports) and Clinician notes into `arlpc_evidence.parquet`.
   Imaging is skipped: PSMA-PET never counts.
2. Run `tasks/arlpc/run_arlpc_extraction.py --provider PROVIDER`: per-note findings, then
   `arlpc_labels.parquet` (one row per cohort patient, `arlpc_`-prefixed columns).
3. Optionally draw the stratified manual-review sample (last cell).

Extraction resumes per note. `REBUILD_LABELS_ONLY` re-applies the gate to the stored findings
without any LLM calls. All run toggles default to `False`; review the printed commands and
paths before enabling a step.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
from IPython.display import display


def find_repo_root(start):
    """Find the repo root (contains pyproject.toml) from the notebook directory."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repo root (pyproject.toml)")


REPO_ROOT = find_repo_root(Path.cwd())
PYTHON = sys.executable


def shell_join(parts):
    return " ".join(shlex.quote(str(part)) for part in parts)


def run_command(parts, env=None):
    print(shell_join(parts))
    process = subprocess.Popen(
        parts, cwd=REPO_ROOT, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, bufsize=0,
    )
    progress_display = None
    buffer = []
    carriage_line = False

    def show_progress(line):
        nonlocal progress_display
        payload = {"text/plain": line}
        if progress_display is None:
            progress_display = display(payload, raw=True, display_id=True)
        else:
            progress_display.update(payload, raw=True)

    while True:
        char = process.stdout.read(1)
        if char == b"":
            break
        if char == b"\r":
            if buffer:
                show_progress(b"".join(buffer).decode("utf-8", errors="replace"))
            buffer = []
            carriage_line = True
        elif char == b"\n":
            line = b"".join(buffer).decode("utf-8", errors="replace")
            if carriage_line:
                show_progress(line)
                progress_display = None
            else:
                print(line)
            buffer = []
            carriage_line = False
        else:
            buffer.append(char)
    if buffer:
        (show_progress if carriage_line else print)(b"".join(buffer).decode("utf-8", errors="replace"))
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"Command failed with exit code {returncode}")


def build_run_env(provider):
    """Child-process environment for the selected provider. Existing environment
    values are preserved except for the explicit notebook parameters below."""
    env = os.environ.copy()
    if provider == "vertex_ai":
        env["VERTEX_PROJECT"] = VERTEX_PROJECT
        env["VERTEX_LOCATION"] = VERTEX_LOCATION
    return env


DEFAULT_DATA_ROOT = Path(
    os.environ.get("LLM_ANNOTATIONS_DATA_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/")
)
PROFILE_DATA_ROOT = Path(os.environ.get("PROFILE_DATA_PATH", "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/"))
DEFAULT_NOTES_PARQUETS = [PROFILE_DATA_ROOT / "CLINICAL_NOTES" / name for name in ("PATHOLOGY_NOTES.parquet", "IMAGING_NOTES.parquet", "PROGRESS_NOTES.parquet")]

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# Provider toggle — everything below adapts to this.
PROVIDER = "vertex_ai"          # or "dfci_gpt"

MODEL_NAME = {
    "dfci_gpt":  "gpt-4o",
    "vertex_ai": "gemini-3.5-flash-lite",
}[PROVIDER]

# Vertex AI settings (unused when PROVIDER == "dfci_gpt")
VERTEX_PROJECT = "gusevlabllm"
VERTEX_LOCATION = "us"  # multi-region: Gemini 3+ is not served in single US regions

MRNS = []                     # pilot subset; the cohort still comes from MRN_FILE
MRN_FILE = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/mrn_lists/icd_prostate_mrn_flags.csv"
NOTES_PARQUET_PATHS = DEFAULT_NOTES_PARQUETS

OUTPUT_DIR = DEFAULT_DATA_ROOT / "LLM_arlpc"
EVIDENCE_PATH = OUTPUT_DIR / "arlpc_evidence.parquet"

# Collection settings
NOTE_TYPES = ["Pathology", "Clinician"]  # Imaging skipped: PSMA-PET never counts
# Trigger window for non-pathology notes; pathology reports are kept whole.
CONTEXT_CHARS = 1500
OVERWRITE_EVIDENCE = False

# LLM extraction settings
MAX_WORKERS = 16
MAX_RETRIES = 3
LIMIT_NOTES = None
OVERWRITE = False

# Manual review sample (last cell)
REVIEW_SEED = 20261118
RUN_REVIEW_SAMPLE = False

# Step toggles
RUN_COLLECT_EVIDENCE = False  # required before extraction
RUN_EXTRACTION = False
RUN_RETRY_FAILURES = False    # re-call only notes whose last attempt failed

RUN_ENV = build_run_env(PROVIDER)

In [ ]:
# True re-applies gates/criteria to the stored findings without LLM calls.
REBUILD_LABELS_ONLY = False

In [ ]:
notes_parquet_status = {path: Path(path).exists() for path in NOTES_PARQUET_PATHS}
evidence_exists = Path(EVIDENCE_PATH).exists()
labels_path = Path(OUTPUT_DIR) / "arlpc_labels.parquet"
labels_exists = labels_path.exists()

print(f"Provider: {PROVIDER}")
print(f"Model: {MODEL_NAME}")
print(f"Parallel workers: {MAX_WORKERS}")
print("PROFILE_DATA note parquets:")
for path, exists in notes_parquet_status.items():
    print(f"  {exists}: {path}")
print(f"Evidence Parquet exists: {evidence_exists}")
print(f"Path: {EVIDENCE_PATH}")
print(f"Labels Parquet exists: {labels_exists}")
print(f"Path: {labels_path}")

In [ ]:
missing_note_parquets = [path for path, exists in notes_parquet_status.items() if not exists]
if missing_note_parquets:
    raise FileNotFoundError(f"Missing PROFILE_DATA note parquets: {missing_note_parquets}")
print("PROFILE_DATA note inputs are ready.")

In [ ]:
collect_evidence_cmd = [
    PYTHON,
    "preprocessing/cli/collect_arlpc_notes.py",
    "--output-dir",
    OUTPUT_DIR,
    "--context-chars",
    str(CONTEXT_CHARS),
]
for notes_parquet_path in NOTES_PARQUET_PATHS:
    collect_evidence_cmd.extend(["--notes-parquet", notes_parquet_path])

if MRNS:
    collect_evidence_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    collect_evidence_cmd.extend(["--mrn-file", MRN_FILE])
if NOTE_TYPES:
    collect_evidence_cmd.extend(["--note-types", *NOTE_TYPES])
if OVERWRITE_EVIDENCE:
    collect_evidence_cmd.append("--overwrite")

print(shell_join(collect_evidence_cmd))

In [ ]:
if RUN_COLLECT_EVIDENCE:
    run_command(collect_evidence_cmd, env=RUN_ENV)
else:
    print("Skipping collect_arlpc_notes.py")

In [ ]:
run_extraction_cmd = [
    PYTHON,
    "tasks/arlpc/run_arlpc_extraction.py",
    "--provider",
    PROVIDER,
    "--output-dir",
    OUTPUT_DIR,
    "--evidence-path",
    EVIDENCE_PATH,
    "--model",
    MODEL_NAME,
    "--max-workers",
    str(MAX_WORKERS),
    "--max-retries",
    str(MAX_RETRIES),
]

if MRNS:
    run_extraction_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
elif MRN_FILE is not None:
    run_extraction_cmd.extend(["--mrn-file", MRN_FILE])
if LIMIT_NOTES is not None:
    run_extraction_cmd.extend(["--limit-notes", str(LIMIT_NOTES)])
if REBUILD_LABELS_ONLY and OVERWRITE:
    raise ValueError("REBUILD_LABELS_ONLY cannot be combined with OVERWRITE")
if REBUILD_LABELS_ONLY:
    run_extraction_cmd.append("--rebuild-labels-only")
if OVERWRITE:
    run_extraction_cmd.append("--overwrite")

print(shell_join(run_extraction_cmd))

In [ ]:
if RUN_EXTRACTION or REBUILD_LABELS_ONLY:
    run_command(run_extraction_cmd, env=RUN_ENV)
else:
    print("Skipping run_arlpc_extraction.py")

In [ ]:
# Re-call only notes whose most recent attempt failed. A plain re-run skips
# every note already in the ledger, failed or not.
retry_cmd = [
    part for part in run_extraction_cmd
    if part not in ("--overwrite", "--rebuild-labels-only")
] + ["--retry-failures"]
print(shell_join(retry_cmd))

if RUN_RETRY_FAILURES:
    run_command(retry_cmd, env=RUN_ENV)
else:
    print("Skipping retry")

In [ ]:
import polars as pl

if labels_path.exists():
    labels = pl.read_parquet(labels_path)
    print(labels["arlpc_review_status"].value_counts().sort("count", descending=True))
    print(labels.group_by("arlpc_strict", "arlpc_broad").len().sort("len", descending=True))
    display(labels.filter(pl.col("arlpc_broad").fill_null(False)).select(
        "DFCI_MRN", "arlpc_strict", "arlpc_strict_date", "arlpc_strict_date_precision",
        "arlpc_strict_evidence_type", "arlpc_strict_quote", "arlpc_broad_date",
        "arlpc_tissue_date", "arlpc_ar_target_markers",
    ).head(20))
else:
    print(f"No labels yet at {labels_path}")


In [ ]:
# Stratified manual-review sample: 20 strict-positive, 10 broad-only, and 20
# negatives that had gate rejections. Each sampled patient is joined to its
# grounded findings, rejected findings and source evidence snippets.
REVIEW_STRATA = {"strict_positive": 20, "broad_only": 10, "negative_with_rejections": 20}
review_path = Path(OUTPUT_DIR) / "arlpc_review_sample.parquet"

if RUN_REVIEW_SAMPLE:
    labels = pl.read_parquet(labels_path)
    strata = {
        "strict_positive": labels.filter(pl.col("arlpc_strict") == True),
        "broad_only": labels.filter((pl.col("arlpc_broad") == True) & (pl.col("arlpc_strict") == False)),
        "negative_with_rejections": labels.filter(
            (pl.col("arlpc_broad") == False) & (pl.col("arlpc_gate_rejections").list.len() > 0)
        ),
    }
    picked = pl.concat([
        frame.sample(n=min(REVIEW_STRATA[name], frame.height), seed=REVIEW_SEED)
             .select("DFCI_MRN").with_columns(pl.lit(name).alias("review_stratum"))
        for name, frame in strata.items()
    ])
    for name, frame in strata.items():
        print(f"{name}: {frame.height} available, {min(REVIEW_STRATA[name], frame.height)} sampled")
    findings = pl.read_parquet(Path(OUTPUT_DIR) / "arlpc_findings.parquet").with_columns(
        pl.lit("grounded").alias("finding_kind"))
    rejected_path = Path(OUTPUT_DIR) / "arlpc_rejected_findings.parquet"
    rejected = (pl.read_parquet(rejected_path).with_columns(pl.lit("rejected").alias("finding_kind"))
                if rejected_path.exists() else None)
    rows = pl.concat([f for f in (findings, rejected) if f is not None], how="diagonal_relaxed")
    evidence = pl.read_parquet(EVIDENCE_PATH).select("note_uid", "snippet")
    sample = (
        picked.join(labels, on="DFCI_MRN", how="left")
              .join(rows, on="DFCI_MRN", how="left")
              .join(evidence, on="note_uid", how="left")
              .sort("review_stratum", "DFCI_MRN", "source_note_date")
    )
    sample.write_parquet(review_path, compression="zstd")
    print(f"Wrote {sample.height} rows for {picked.height} patients to {review_path}")
else:
    print("Skipping review sample")
